# ASR Demo — Listen, Attend and Spell

Этот ноутбук демонстрирует инференс обученной LAS модели для распознавания речи.

**Как использовать:**
1. Запустите ячейки по порядку
2. Для проверки на своих данных — вставьте ссылку на Google Drive с датасетом в нужном формате

## 1. Установка

In [ ]:
!git clone https://github.com/<your_username>/asr_project.git
%cd asr_project
!pip install -r requirements.txt -q

## 2. Скачивание весов модели

In [ ]:
!pip install gdown -q
!mkdir -p saved
# замените ссылку на свою
!gdown <GOOGLE_DRIVE_LINK_TO_WEIGHTS> -O saved/model_best.pth

## 3. Инференс на LibriSpeech test-clean

In [ ]:
!python inference.py \
  inferencer.from_pretrained=saved/model_best.pth \
  datasets=example_eval \
  inferencer.save_path=inference_output

## 4. Подсчёт метрик

Для подсчёта CER/WER нужен отдельный скрипт `calc_metrics.py`, который сравнивает предсказания с ground truth.

In [ ]:
# Подсчёт метрик на test-clean (если gt доступны через inference)
# Предсказания сохраняются в data/saved/inference_output/test-clean/
# GT для LibriSpeech берутся из датасета
import json
from pathlib import Path

# соберём GT из index файла LibriSpeech
index_path = Path("data/datasets/librispeech/test-clean_index.json")
if index_path.exists():
    with open(index_path) as f:
        index = json.load(f)
    gt_dir = Path("gt_transcriptions")
    gt_dir.mkdir(exist_ok=True)
    for entry in index:
        audio_id = Path(entry["path"]).stem
        with open(gt_dir / f"{audio_id}.txt", "w") as f:
            f.write(entry["text"])
    print(f"Saved {len(index)} GT transcriptions")
else:
    print("Index not found, run inference first to download data")

In [ ]:
!python calc_metrics.py \
  --gt-dir gt_transcriptions \
  --pred-dir data/saved/inference_output/test-clean

## 5. Инференс на своих данных

Вставьте ссылку на Google Drive с папкой в формате:
```
my_dataset/
├── audio/
│   ├── utt1.wav
│   └── utt2.wav
└── transcriptions/  (опционально)
    ├── utt1.txt
    └── utt2.txt
```

In [ ]:
#@title Вставьте ссылку на Google Drive с датасетом { display-mode: "form" }
gdrive_link = ""  #@param {type:"string"}

if gdrive_link:
    !pip install gdown -q
    !gdown --folder "{gdrive_link}" -O custom_dataset --remaining-ok
    print("Dataset downloaded")
else:
    print("Вставьте ссылку на Google Drive в поле выше")

In [ ]:
import os

if os.path.exists("custom_dataset/audio"):
    audio_dir = "custom_dataset/audio"
    transc_dir = "custom_dataset/transcriptions" if os.path.exists("custom_dataset/transcriptions") else "null"

    !python inference.py \
      datasets=custom_dir \
      datasets.test.audio_dir={audio_dir} \
      datasets.test.transcription_dir={transc_dir} \
      inferencer.from_pretrained=saved/model_best.pth \
      inferencer.save_path=custom_output

    if transc_dir != "null":
        !python calc_metrics.py \
          --gt-dir {transc_dir} \
          --pred-dir data/saved/custom_output/test
else:
    print("Сначала загрузите датасет (ячейка выше)")

## 6. Примеры предсказаний

In [ ]:
from pathlib import Path

pred_dir = Path("data/saved/inference_output/test-clean")
if pred_dir.exists():
    pred_files = sorted(pred_dir.glob("*.txt"))[:10]
    for pf in pred_files:
        print(f"{pf.stem}: {pf.read_text().strip()}")
else:
    print("Сначала запустите инференс")